# BIRDI JEPA Retrieval-Augmented Typed-IR Training on Kaggle GPU

This notebook runs the practical JEPA typed algorithm-IR experiment:

1. Load `Instruction`, `Input`, and reference `Output` code.
2. Build pseudo-label semantic state/data-flow graphs and typed algorithm IR from the reference code.
3. Encode problem text and target typed IR with a frozen Qwen3 embedding model.
4. Train a JEPA predictor from problem latent to typed-IR latent with contrastive and anti-collapse losses.
5. Retrieve top-k typed IR prototypes from the predicted JEPA latent.
6. Train a CodeT5+ IR repair decoder conditioned on `problem text + retrieved IR + z_pred`.
7. Decode typed IR, then compile that IR directly into Python code without a final code LLM.

The retrieved IR examples are automatically extracted from the training solutions; no hand-written graph database is required. The final code stage is deterministic IR-to-Python compilation, so Qwen or another code LLM cannot bypass the IR and solve the problem directly.


In [ ]:
# Kaggle setup. Run this once at the top of the notebook.
from pathlib import Path
import os
import sys
import shutil
import subprocess

# Text-only project. transformers can eagerly import the optional timm/torchvision
# stack while enumerating its model mappings. On Kaggle, timm is present without a
# matching torchvision, which crashes an otherwise text-only run. The birdi_jepa
# package disables that optional import path at runtime (a real guard, unlike the
# TRANSFORMERS_NO_* env vars which transformers does not honour), so we leave the
# base image's torch / torchvision / torchaudio untouched instead of uninstalling
# them and creating an inconsistent environment.

IS_KAGGLE = Path('/kaggle/input').exists()
INPUT_ROOT = Path('/kaggle/input') if IS_KAGGLE else Path.cwd()
WORK_ROOT = Path('/kaggle/working') if IS_KAGGLE else Path.cwd()
PKG_DST = WORK_ROOT / 'birdi_jepa'

print('Input root:', INPUT_ROOT)
if IS_KAGGLE:
    print('Visible Kaggle inputs:')
    for path in sorted(INPUT_ROOT.iterdir()):
        print(' -', path)

def has_package_files(path):
    return (path / 'state_graph.py').exists() and (path / 'training.py').exists()

# Kaggle datasets are read-only, so copy the uploaded package files into /kaggle/working.
local_pkg = Path.cwd() / 'birdi_jepa'
pkg_markers = [p for p in INPUT_ROOT.rglob('state_graph.py') if has_package_files(p.parent)]
if IS_KAGGLE and pkg_markers:
    PKG_SRC = pkg_markers[0].parent
    print('Using birdi_jepa source:', PKG_SRC)
    PKG_DST.mkdir(parents=True, exist_ok=True)
    for file in PKG_SRC.glob('*.py'):
        shutil.copy2(file, PKG_DST / file.name)
    PROJECT_DIR = WORK_ROOT
elif has_package_files(local_pkg):
    PROJECT_DIR = Path.cwd()
elif pkg_markers:
    PKG_SRC = pkg_markers[0].parent
    print('Using birdi_jepa source:', PKG_SRC)
    PROJECT_DIR = PKG_SRC.parent
else:
    print('Could not find package files. Visible files include:')
    for path in list(INPUT_ROOT.rglob('*'))[:120]:
        print(path)
    raise FileNotFoundError('Could not find birdi_jepa package files: state_graph.py and training.py')

sys.path.insert(0, str(PROJECT_DIR))

# Find the CSV once and keep DATA_PATH global for the rest of the notebook.
csv_matches = list(INPUT_ROOT.rglob('Python Programming Questions Dataset.csv'))
if not csv_matches:
    csv_matches = list(Path.cwd().rglob('Python Programming Questions Dataset.csv'))
if not csv_matches:
    print('Could not find the CSV. Visible files include:')
    for path in list(INPUT_ROOT.rglob('*'))[:120]:
        print(path)
    raise FileNotFoundError('Could not find Python Programming Questions Dataset.csv')
DATA_PATH = csv_matches[0]
print('Using CSV:', DATA_PATH)

REQ = PROJECT_DIR / 'requirements-kaggle.txt'
if REQ.exists():
    install_cmd = [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REQ)]
else:
    install_cmd = [sys.executable, '-m', 'pip', 'install', '-q', 'transformers==4.51.3', 'tqdm']

try:
    subprocess.check_call(install_cmd)
except subprocess.CalledProcessError as exc:
    print('Package install failed. If Kaggle internet is disabled, enable it or attach cached wheels/models.')
    print('Continuing with the currently installed environment. Error:', exc)

from birdi_jepa.training import prepare_dataframe
from birdi_jepa.state_graph import graph_from_code

print('Project directory:', PROJECT_DIR)
print('Setup complete. birdi_jepa imports work.')


In [ ]:
import gc
import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

from birdi_jepa.algorithm_ir import compile_ir_to_python
from birdi_jepa.decoder import DEFAULT_DECODER_MODEL, generate_graph_json, train_latent_decoder
from birdi_jepa.embedding import DEFAULT_EMBED_MODEL, encode_texts
from birdi_jepa.training import (
    embedding_diagnostics,
    prepare_dataframe,
    nearest_indices,
    predict_embeddings,
    retrieval_metrics,
    train_jepa_predictor,
)
from birdi_jepa.state_graph import build_problem_text, graph_from_code

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# Configuration: full-data retrieval-augmented CodeT5+ typed-IR repair run.
RUN_TRAIN_FROM_SCRATCH = True
RUN_DECODER_TRAINING = True
RUN_INFERENCE_EXAMPLES = True
RUN_SAVE_MODELS = True
RUN_LOAD_SAVED_MODELS = False
USE_SAVED_OUTPUT_220M = False

TARGET_REPRESENTATION = 'typed_algorithm_ir'
MAX_ROWS = None
MIN_CONFIDENCE = 0.75
# Rare algorithm families that random sampling under-represents. With MAX_ROWS=None,
# all valid rows are kept; these are retained for capped smoke tests.
FOCUS_KEYWORDS = [
    'gcd', 'greatest common divisor', 'common divisor', 'gcf',
    'lcm', 'least common multiple', 'fibonacci', 'factorial',
    'palindrome', 'anagram', 'binary search',
    'average', 'mean', 'negative', 'positive',
]
RETRIEVAL_TOP_K = 10
RETRIEVAL_PRINT_K = 3

# Retrieval-augmented decoder context.
DECODER_CONTEXT_TOP_K = 2
DECODER_CONTEXT_CANDIDATE_K = 10
DECODER_CONTEXT_IR_CHARS = 900
DECODER_MAX_TRAIN_ROWS = 4096
DECODER_MAX_VAL_ROWS = 256

# Frozen embedding model used for both problem text and typed-IR target latents.
EMBED_MODEL = DEFAULT_EMBED_MODEL
EMBED_BATCH_SIZE = 4
PROBLEM_PROMPT = 'Instruct: Given a programming problem, retrieve the matching typed algorithm IR.' + chr(10) + 'Query: '
TARGET_PROMPT = 'Instruct: Represent this typed algorithm IR for matching to a programming problem.' + chr(10) + 'IR: '

JEPA_EPOCHS = 20
JEPA_BATCH_SIZE = 128
JEPA_LR = 1e-3
CONTRASTIVE_WEIGHT = 1.0
TEMPERATURE = 0.03
VICREG_VARIANCE_WEIGHT = 0.05
VICREG_COVARIANCE_WEIGHT = 0.005
VICREG_VARIANCE_TARGET = 0.02

# Typed-IR repair decoder: CodeT5+ 770M conditioned on problem text + retrieved IR
# + predicted typed-IR latent. 4096 rows/top-2 context/640 tokens has completed
# in about 2.5 hours on Kaggle T4 x2 in previous runs.
DECODER_MODEL = 'Salesforce/codet5p-770m'
DECODER_EPOCHS = 1
DECODER_BATCH_SIZE = 1
DECODER_OPTIMIZER = 'adafactor'
DECODER_LR = 3e-5
DECODER_WEIGHT_DECAY = 0.0
DECODER_MAX_GRAD_NORM = 1.0
DECODER_PREFIX_TOKENS = 16
FREEZE_DECODER_BASE = False
DECODER_GRADIENT_ACCUMULATION_STEPS = 4
DECODER_GRADIENT_CHECKPOINTING = True
DECODER_SOURCE_PREFIX = (
    'Repair/adapt the retrieved typed algorithm IR examples into one correct typed algorithm IR for the problem. '
    'Use fixed operators when possible: FUNC, FILTER, MEAN, SUM, COUNT, SORT, MIN, MAX, GCD, LCM, FOR, WHILE, IF, ELSE, END, RETURN. '
    'Use the retrieved IR contexts as imperfect prototypes, but change conditions, operators, variables, and control flow when the problem requires it. '
    'Output only typed IR lines. Do not output JSON. Do not output Python code.' + chr(10)
)
GRAPH_MAX_SOURCE_LENGTH = 640
GRAPH_MAX_TARGET_LENGTH = 640
GRAPH_MAX_NEW_TOKENS = 640

# Final IR -> code stage. This is a deterministic compiler, not Qwen/code-LLM generation.
CODEGEN_FUNCTION_NAME = 'solution'

# The setup cell already searches recursively and sets DATA_PATH. Keep that value
# because Kaggle may mount datasets under nested paths such as
# /kaggle/input/datasets/<owner>/<dataset>/...
if 'DATA_PATH' not in globals() or DATA_PATH is None or not Path(DATA_PATH).exists():
    DATA_CANDIDATES = [
        Path('/kaggle/input/python-programming-questions-dataset/Python Programming Questions Dataset.csv'),
        Path('/kaggle/input/Python Programming Questions Dataset.csv'),
        Path.cwd() / 'Python Programming Questions Dataset.csv',
    ]
    csv_matches = []
    if INPUT_ROOT.exists():
        csv_matches = list(INPUT_ROOT.rglob('Python Programming Questions Dataset.csv'))
    if not csv_matches:
        csv_matches = [p for p in DATA_CANDIDATES if p.exists()]
    if not csv_matches:
        print('Could not find the CSV. Visible input files include:')
        for path in list(INPUT_ROOT.rglob('*'))[:120]:
            print(' -', path)
        raise FileNotFoundError('Could not find Python Programming Questions Dataset.csv. Add it as a Kaggle input dataset.')
    DATA_PATH = csv_matches[0]
CACHE_DIR = WORK_ROOT / 'birdi_cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print('Dataset:', DATA_PATH)
print('Rows:', 'full dataset' if MAX_ROWS is None else MAX_ROWS)
print('Target representation:', TARGET_REPRESENTATION)
print('Embedding model:', EMBED_MODEL)
print('Typed-IR repair decoder:', DECODER_MODEL)
print('IR->code:', 'symbolic typed-IR compiler (no final Qwen/code LLM)')
print('Train from scratch:', RUN_TRAIN_FROM_SCRATCH)
print('Retrieval top-k:', RETRIEVAL_TOP_K, '| decoder context top-k:', DECODER_CONTEXT_TOP_K)
print('Decoder train/val row budget:', DECODER_MAX_TRAIN_ROWS, '/', DECODER_MAX_VAL_ROWS)
print('Decoder optimizer:', DECODER_OPTIMIZER, 'lr:', DECODER_LR, 'epochs:', DECODER_EPOCHS)
print('Decoder batch/accum:', DECODER_BATCH_SIZE, '/', DECODER_GRADIENT_ACCUMULATION_STEPS)
print('Decoder source length/target length:', GRAPH_MAX_SOURCE_LENGTH, '/', GRAPH_MAX_TARGET_LENGTH)


## Stage A: Build Pseudo-Label Semantic Graphs and Typed Algorithm IR


In [ ]:
prepared = prepare_dataframe(
    DATA_PATH,
    max_rows=MAX_ROWS,
    seed=SEED,
    min_confidence=MIN_CONFIDENCE,
    always_include_keywords=FOCUS_KEYWORDS,
)
df = prepared.frame
target_texts = prepared.ir_texts

print('Prepared rows:', len(df))
display(df[['Instruction', 'Input', 'pseudo_label_confidence', 'algorithm_ir_confidence', 'algorithm_ir_parse_ok', 'state_graph_nodes', 'state_graph_edges']].head())
display(df[['pseudo_label_confidence', 'algorithm_ir_confidence', 'state_graph_nodes', 'state_graph_edges']].describe())
print('Graph parse success rate:', round(df['pseudo_label_parse_ok'].mean(), 4))
print('IR parse success rate:', round(df['algorithm_ir_parse_ok'].mean(), 4))


In [ ]:
sample_idx = int(df['algorithm_ir_confidence'].idxmax())
print('Problem text:\n', prepared.problem_texts[sample_idx])
print('\nReference code:\n', df.loc[sample_idx, 'Output'])
print('\nTyped algorithm IR:')
print(prepared.ir_texts[sample_idx])
print('\nBackup semantic state graph JSON:')
print(json.dumps(json.loads(prepared.graph_texts[sample_idx]), indent=2)[:1800])


## Stage B: Frozen Problem and Typed-IR Embeddings

Here the JEPA target is the typed algorithm IR rather than loose graph JSON. The old graph JSON is still available for diagnostics, but the learned target space, retrieval index, decoder target, and final compiler now use typed IR.


In [ ]:
safe_model_name = EMBED_MODEL.replace('/', '__').replace('-', '_')
problem_cache = CACHE_DIR / f'problem_embeddings_{safe_model_name}_{len(df)}.npy'
target_cache = CACHE_DIR / f'target_ir_embeddings_{safe_model_name}_{len(df)}.npy'

problem_embeddings = encode_texts(
    prepared.problem_texts,
    model_name=EMBED_MODEL,
    batch_size=EMBED_BATCH_SIZE,
    prompt=PROBLEM_PROMPT,
    cache_path=problem_cache,
)
target_embeddings = encode_texts(
    target_texts,
    model_name=EMBED_MODEL,
    batch_size=EMBED_BATCH_SIZE,
    prompt=TARGET_PROMPT,
    cache_path=target_cache,
)

print('Problem embeddings:', problem_embeddings.shape, problem_embeddings.dtype)
print('Target typed-IR embeddings:', target_embeddings.shape, target_embeddings.dtype)
print('Caches:', problem_cache, target_cache)

for name, emb in [('problem', problem_embeddings), ('target_ir', target_embeddings)]:
    diag = embedding_diagnostics(emb, sample_size=min(512, len(emb)), seed=SEED)
    print()
    print('Embedding diagnostics:', name)
    display(pd.Series(diag))
    if diag['offdiag_cos_mean'] > 0.95:
        print('WARNING: embedding set is nearly collapsed. Downstream decoded outputs may repeat.')

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## Stage C: Train JEPA Predictor

Train a small predictor `P(problem_embedding) -> typed_ir_embedding`. Contrastive loss is weighted strongly here because the core failure to avoid is collapse into one generic IR region.


In [ ]:
predictor, history, split = train_jepa_predictor(
    problem_embeddings=problem_embeddings,
    target_embeddings=target_embeddings,
    epochs=JEPA_EPOCHS,
    batch_size=JEPA_BATCH_SIZE,
    lr=JEPA_LR,
    contrastive_weight=CONTRASTIVE_WEIGHT,
    temperature=TEMPERATURE,
    variance_weight=VICREG_VARIANCE_WEIGHT,
    covariance_weight=VICREG_COVARIANCE_WEIGHT,
    variance_target=VICREG_VARIANCE_TARGET,
    val_fraction=0.1,
    seed=SEED,
)

display(history)
history.to_csv(CACHE_DIR / 'jepa_history.csv', index=False)
torch.save(predictor.state_dict(), CACHE_DIR / 'jepa_predictor.pt')
print('Saved JEPA artifacts to:', CACHE_DIR)


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 9))
axes = axes.ravel()

axes[0].plot(history['epoch'], history['train_loss'], label='train')
axes[0].plot(history['epoch'], history['val_loss'], label='validation')
axes[0].set_title('Total JEPA loss')
axes[0].set_xlabel('epoch')
axes[0].legend()

axes[1].plot(history['epoch'], history['val_jepa'], label='latent MSE')
axes[1].plot(history['epoch'], history['val_contrastive'], label='contrastive')
axes[1].plot(history['epoch'], history['val_variance'], label='variance')
axes[1].plot(history['epoch'], history['val_covariance'], label='covariance')
axes[1].set_title('Validation loss components')
axes[1].set_xlabel('epoch')
axes[1].legend()

axes[2].plot(history['epoch'], history['mean_cosine'], label='matched cosine')
axes[2].plot(history['epoch'], history['retrieval_at_1'], label='retrieval@1')
axes[2].plot(history['epoch'], history['retrieval_at_5'], label='retrieval@5')
axes[2].set_title('Retrieval metrics')
axes[2].set_xlabel('epoch')
axes[2].legend()

axes[3].plot(history['epoch'], history['pred_dim_std_mean'], label='dim std mean')
axes[3].plot(history['epoch'], history['pred_offdiag_cos_mean'], label='offdiag cosine mean')
axes[3].axhline(VICREG_VARIANCE_TARGET, color='gray', linestyle='--', linewidth=1, label='variance target')
axes[3].set_title('Collapse diagnostics')
axes[3].set_xlabel('epoch')
axes[3].legend()

plt.tight_layout()
plt.show()


In [ ]:
predicted_latents = predict_embeddings(predictor, problem_embeddings)
val_idx = split['val_idx']

with torch.no_grad():
    val_metrics = retrieval_metrics(
        torch.tensor(predicted_latents[val_idx]),
        torch.tensor(target_embeddings[val_idx]),
    )
print('Final validation metrics:', val_metrics)

for name, emb in [('target_ir', target_embeddings), ('predicted_ir', predicted_latents)]:
    diag = embedding_diagnostics(emb, sample_size=min(512, len(emb)), seed=SEED)
    print()
    print('Embedding diagnostics:', name)
    display(pd.Series(diag))
    if diag['offdiag_cos_mean'] > 0.95 or diag['dim_std_mean'] < 0.005:
        print('WARNING: this latent set may be collapsed. Check retrieval diversity before trusting Stage D/E.')

np.save(CACHE_DIR / 'predicted_ir_latents.npy', predicted_latents)
df.to_csv(CACHE_DIR / 'prepared_typed_ir_labeled_data.csv', index=False)


In [ ]:
# Nearest-neighbor sanity check in the typed-IR latent space.
all_target_norm = target_embeddings / np.maximum(np.linalg.norm(target_embeddings, axis=1, keepdims=True), 1e-12)
pred_norm = predicted_latents / np.maximum(np.linalg.norm(predicted_latents, axis=1, keepdims=True), 1e-12)

for idx in val_idx[:3]:
    sims = pred_norm[idx] @ all_target_norm.T
    best = int(np.argmax(sims))
    print('=' * 100)
    print('Validation row:', int(idx), '| nearest IR row:', best, '| hit:', best == int(idx), '| cosine:', round(float(sims[best]), 4))
    print()
    print('Problem:')
    print(prepared.problem_texts[int(idx)])
    print()
    print('True typed IR:')
    print(prepared.ir_texts[int(idx)][:1200])
    print()
    print('Nearest predicted typed IR:')
    print(prepared.ir_texts[best][:1200])


## Stage C+: Pairwise Collapse Check

This cell tests the exact failure you noticed. Similar problems should have higher predicted typed-IR latent similarity than unrelated problems. If every score is `+1.0000`, the predictor has collapsed.


In [ ]:
PAIR_A = {'statement': 'Remove all positive numbers from a list.', 'input': '[-2, 3, -1, 4]'}
PAIR_B = {'statement': 'Filter out the positive integers from a list.', 'input': '[-5, 1, 0, 6]'}
PAIR_C = {'statement': 'Sort a list of numbers in ascending order.', 'input': '[3, 1, 2]'}

def cos(a, b):
    a = np.asarray(a, dtype='float32'); b = np.asarray(b, dtype='float32')
    return float((a @ b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))

pair_texts = [build_problem_text(p['statement'], p['input']) for p in [PAIR_A, PAIR_B, PAIR_C]]
pair_emb = encode_texts(pair_texts, model_name=EMBED_MODEL, batch_size=EMBED_BATCH_SIZE,
                        prompt=PROBLEM_PROMPT, cache_path=None)
pair_z = predict_embeddings(predictor, pair_emb)
print('A:', pair_texts[0])
print('B:', pair_texts[1])
print('C:', pair_texts[2])
print('-' * 80)
print(f'similar A-B predicted typed-IR latent cosine:   {cos(pair_z[0], pair_z[1]):+.4f}')
print(f'different A-C predicted typed-IR latent cosine: {cos(pair_z[0], pair_z[2]):+.4f}')
print(f'different B-C predicted typed-IR latent cosine: {cos(pair_z[1], pair_z[2]):+.4f}')


## Stage D: Train Retrieval-Augmented Typed-IR Repair Decoder

For each training row we first retrieve similar typed IR prototypes from the JEPA-predicted IR latent, excluding the same row to avoid answer copying. CodeT5+ then learns:

`problem text + top-k retrieved typed IR + z_pred -> corrected target typed IR`


In [ ]:
def is_valid_typed_ir(text):
    lines = [line.strip() for line in str(text).splitlines() if line.strip()]
    if not lines:
        return False
    has_func = any(line.startswith('FUNC ') for line in lines[:2])
    has_return = any(line.startswith('RETURN ') for line in lines)
    bad_json = str(text).lstrip().startswith(('{', '['))
    bad_python = any(line.startswith('def ') for line in lines)
    return has_func and has_return and not bad_json and not bad_python

def compact_ir_for_context(ir_text, max_chars=DECODER_CONTEXT_IR_CHARS):
    text = str(ir_text).strip()
    if len(text) <= max_chars:
        return 'TYPED_IR\n' + text
    return 'TYPED_IR\n' + text[:max_chars].rstrip() + '\n...'

def ir_has_function(ir_text):
    return any(line.strip().startswith('FUNC ') for line in str(ir_text).splitlines())

def rerank_context_indices(row_indices):
    candidates = [int(idx) for idx in row_indices[:DECODER_CONTEXT_CANDIDATE_K]]
    function_like = [idx for idx in candidates if ir_has_function(prepared.ir_texts[idx])]
    remainder = [idx for idx in candidates if idx not in function_like]
    return (function_like + remainder)[:DECODER_CONTEXT_TOP_K]

def choose_retrieved_ir(row_indices):
    chosen = rerank_context_indices(row_indices)
    if not chosen:
        return [], 'no retrieval context'
    reason = 'function IR preferred' if ir_has_function(prepared.ir_texts[chosen[0]]) else 'top cosine IR used'
    return chosen, reason

def build_decoder_source(problem_text, retrieved_indices):
    parts = [problem_text, '', 'Retrieved typed IR prototypes:']
    for rank, idx in enumerate(retrieved_indices, start=1):
        parts.append(f'--- prototype {rank} row={int(idx)} ---')
        parts.append(compact_ir_for_context(prepared.ir_texts[int(idx)]))
    parts.append('')
    parts.append('Return the corrected typed algorithm IR for the problem.')
    return '\n'.join(parts)

retrieved_idx, retrieved_scores = nearest_indices(
    predicted_latents,
    target_embeddings,
    top_k=DECODER_CONTEXT_CANDIDATE_K,
    exclude_self=True,
)

decoder_source_texts = []
decoder_context_rows = []
for row_id, problem_text in enumerate(prepared.problem_texts):
    context_indices = rerank_context_indices(retrieved_idx[row_id])
    decoder_context_rows.append(context_indices)
    decoder_source_texts.append(build_decoder_source(problem_text, context_indices))

print('Built retrieval-augmented decoder sources:', len(decoder_source_texts))
print('Example decoder source:')
print(decoder_source_texts[0][:1600])
print('\nTarget typed IR:')
print(prepared.ir_texts[0][:1200])

def select_decoder_indices(indices, max_count, seed=SEED):
    indices = np.asarray(indices, dtype=np.int64)
    if max_count is None or len(indices) <= max_count:
        return indices
    rng = np.random.default_rng(seed)
    keyword_hits = []
    for idx in indices:
        text = (prepared.problem_texts[int(idx)] + '\n' + prepared.ir_texts[int(idx)]).lower()
        if any(keyword in text for keyword in ['gcd', 'hcf', 'lcm', 'average', 'mean', 'negative', 'positive', 'filter', 'pangram']):
            keyword_hits.append(int(idx))
    keyword_hits = list(dict.fromkeys(keyword_hits))
    remaining = [int(idx) for idx in indices if int(idx) not in set(keyword_hits)]
    keep_keyword = keyword_hits[: min(len(keyword_hits), max_count // 2)]
    fill = max_count - len(keep_keyword)
    if fill > 0 and remaining:
        sampled = rng.choice(remaining, size=min(fill, len(remaining)), replace=False).tolist()
    else:
        sampled = []
    chosen = keep_keyword + sampled
    rng.shuffle(chosen)
    return np.asarray(chosen, dtype=np.int64)

decoder_train_idx = select_decoder_indices(split['train_idx'], DECODER_MAX_TRAIN_ROWS, seed=SEED)
decoder_val_idx = select_decoder_indices(split['val_idx'], DECODER_MAX_VAL_ROWS, seed=SEED + 1)
print('Decoder train rows:', len(decoder_train_idx), '| val rows:', len(decoder_val_idx))
print('Approx decoder optimizer steps:', int(np.ceil(len(decoder_train_idx) / max(1, DECODER_GRADIENT_ACCUMULATION_STEPS))) * DECODER_EPOCHS)

if RUN_DECODER_TRAINING:
    graph_decoder = train_latent_decoder(
        problem_texts=decoder_source_texts,
        graph_texts=target_texts,
        predicted_latents=predicted_latents,
        train_idx=decoder_train_idx,
        val_idx=decoder_val_idx,
        checkpoint=DECODER_MODEL,
        epochs=DECODER_EPOCHS,
        batch_size=DECODER_BATCH_SIZE,
        lr=DECODER_LR,
        weight_decay=DECODER_WEIGHT_DECAY,
        max_source_length=GRAPH_MAX_SOURCE_LENGTH,
        max_target_length=GRAPH_MAX_TARGET_LENGTH,
        source_prefix=DECODER_SOURCE_PREFIX,
        num_prefix_tokens=DECODER_PREFIX_TOKENS,
        freeze_base=FREEZE_DECODER_BASE,
        gradient_accumulation_steps=DECODER_GRADIENT_ACCUMULATION_STEPS,
        enable_gradient_checkpointing=DECODER_GRADIENT_CHECKPOINTING,
        optimizer_name=DECODER_OPTIMIZER,
        max_grad_norm=DECODER_MAX_GRAD_NORM,
        seed=SEED,
    )
    display(graph_decoder.history)
    graph_decoder.history.to_csv(CACHE_DIR / 'typed_ir_decoder_history.csv', index=False)
    torch.save(graph_decoder.model.state_dict(), CACHE_DIR / 'typed_ir_decoder.pt')
    print('Saved typed-IR decoder artifacts to:', CACHE_DIR)
else:
    print('RUN_DECODER_TRAINING=False, Stage D decoder training skipped.')


In [ ]:
if 'graph_decoder' not in globals():
    print('No typed-IR decoder found. Run Stage D training or load a compatible saved model first.')
else:
    decode_idx = [int(i) for i in decoder_val_idx[:5]]
    pred_irs = generate_graph_json(
        graph_decoder,
        decoder_source_texts,
        predicted_latents,
        decode_idx,
        max_source_length=GRAPH_MAX_SOURCE_LENGTH,
        max_new_tokens=GRAPH_MAX_NEW_TOKENS,
        source_prefix=DECODER_SOURCE_PREFIX,
    )

    valid_count = sum(is_valid_typed_ir(text) for text in pred_irs)
    print(f'Valid generated typed IR examples: {valid_count}/{len(pred_irs)}')

    for idx, ir_text in zip(decode_idx, pred_irs):
        print('=' * 100)
        print('Problem:')
        print(prepared.problem_texts[int(idx)])
        print()
        print('Retrieved repair context prefix:')
        print(decoder_source_texts[int(idx)][:900])
        print()
        print('True typed IR prefix:')
        print(prepared.ir_texts[int(idx)][:900])
        print()
        print('Decoded typed IR prefix:')
        print(str(ir_text)[:900])
        print()
        print('Compiled code from decoded typed IR:')
        print(compile_ir_to_python(ir_text, function_name='debug_solution'))


## Stage D+: Save Trained Models for Reuse

This persists the trained JEPA predictor and typed-IR decoder under `/kaggle/working/birdi_cache/saved_models`. Commit the Kaggle notebook output or turn this folder into a Kaggle Dataset if you want to load it in a later session.


In [ ]:
# SAVE TRAINED MODELS FOR REUSE. Run after Stage C and Stage D finish.
import json as _json

if RUN_SAVE_MODELS and 'graph_decoder' in globals():
    SAVE_DIR = CACHE_DIR / 'saved_models'
    SAVE_DIR.mkdir(parents=True, exist_ok=True)

    torch.save(predictor.state_dict(), SAVE_DIR / 'jepa_predictor.pt')
    torch.save(graph_decoder.model.state_dict(), SAVE_DIR / 'typed_ir_decoder.pt')
    graph_decoder.tokenizer.save_pretrained(SAVE_DIR / 'decoder_tokenizer')

    meta = {
        'predictor': {
            'input_dim': int(problem_embeddings.shape[1]),
            'target_dim': int(target_embeddings.shape[1]),
            'hidden_dim': 1024,
            'dropout': 0.1,
            'contrastive_weight': CONTRASTIVE_WEIGHT,
            'temperature': TEMPERATURE,
            'variance_weight': VICREG_VARIANCE_WEIGHT,
            'covariance_weight': VICREG_COVARIANCE_WEIGHT,
            'variance_target': VICREG_VARIANCE_TARGET,
        },
        'decoder': {
            'checkpoint': DECODER_MODEL,
            'latent_dim': int(predicted_latents.shape[1]),
            'num_prefix_tokens': DECODER_PREFIX_TOKENS,
            'source_prefix': DECODER_SOURCE_PREFIX,
            'max_source_length': GRAPH_MAX_SOURCE_LENGTH,
            'max_target_length': GRAPH_MAX_TARGET_LENGTH,
            'max_new_tokens': GRAPH_MAX_NEW_TOKENS,
            'context_top_k': DECODER_CONTEXT_TOP_K,
            'context_candidate_k': DECODER_CONTEXT_CANDIDATE_K,
            'context_ir_chars': DECODER_CONTEXT_IR_CHARS,
            'decoder_max_train_rows': DECODER_MAX_TRAIN_ROWS,
            'decoder_max_val_rows': DECODER_MAX_VAL_ROWS,
            'retrieval_augmented_repair': True,
            'target_representation': TARGET_REPRESENTATION,
        },
        'embed_model': EMBED_MODEL,
        'problem_prompt': PROBLEM_PROMPT,
        'target_prompt': TARGET_PROMPT,
        'trained_from_scratch': RUN_TRAIN_FROM_SCRATCH,
    }
    (SAVE_DIR / 'birdi_meta.json').write_text(_json.dumps(meta, indent=2))

    print('Saved models to:', SAVE_DIR)
    for p in sorted(SAVE_DIR.rglob('*')):
        if p.is_file():
            print(f'  {p.relative_to(SAVE_DIR)} ({p.stat().st_size / 1e6:.1f} MB)')
else:
    print('Skipping model save. RUN_SAVE_MODELS:', RUN_SAVE_MODELS, '| graph_decoder exists:', 'graph_decoder' in globals())


## Optional: Load Saved Models

For a later Kaggle session, add the committed output/dataset that contains `saved_models`, set `LOAD_DIR` to that mounted folder, switch `RUN_LOAD_SAVED_MODELS` to `True`, then run this cell after setup/import/config. After loading, you can skip the Stage C and Stage D training cells.


In [ ]:
# Optional loader. Default is controlled in the configuration cell.
LOAD_DIR = CACHE_DIR / 'saved_models'
# Example for a later Kaggle session:
# LOAD_DIR = Path('/kaggle/input/<your-saved-model-dataset>/saved_models')

if RUN_LOAD_SAVED_MODELS:
    import json as _json
    import pandas as _pd
    from transformers import AutoTokenizer
    from birdi_jepa.training import make_predictor
    from birdi_jepa.decoder import DecoderArtifacts, build_latent_prefix_model

    meta = _json.loads((LOAD_DIR / 'birdi_meta.json').read_text())
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    pc = meta['predictor']
    predictor = make_predictor(
        pc['input_dim'],
        pc['target_dim'],
        hidden_dim=pc.get('hidden_dim', 1024),
        dropout=pc.get('dropout', 0.1),
    )
    predictor.load_state_dict(torch.load(LOAD_DIR / 'jepa_predictor.pt', map_location=device))
    predictor.to(device).eval()

    dc = meta['decoder']
    _decoder_model = build_latent_prefix_model(
        checkpoint=dc['checkpoint'],
        latent_dim=dc['latent_dim'],
        num_prefix_tokens=dc['num_prefix_tokens'],
    )
    _decoder_model.load_state_dict(torch.load(LOAD_DIR / 'typed_ir_decoder.pt', map_location=device))
    _decoder_model.to(device).eval()
    _tokenizer = AutoTokenizer.from_pretrained(str(LOAD_DIR / 'decoder_tokenizer'))
    graph_decoder = DecoderArtifacts(model=_decoder_model, tokenizer=_tokenizer, history=_pd.DataFrame())

    print('Loaded predictor and typed-IR decoder from:', LOAD_DIR)
else:
    print('Model loading skipped. This notebook is configured to train from scratch.')


## Stage E: Four-Problem Inference Probe

This cell uses four hand-written probe problems: two intentionally similar tasks and two different tasks. For each one it prints the JEPA cross-similarity matrix, top-k retrieved typed IR examples, decoded IR, final deterministic Python code, and simple execution probes. These prompts are not copied from the training CSV, but the nearest training rows are shown so overlap is visible.


In [ ]:
INFERENCE_PROBLEMS = [
    {
        'tag': 'A',
        'statement': 'Given two positive integers a and b, compute their greatest common divisor using the Euclidean algorithm.',
        'input': 'a = 84, b = 30',
    },
    {
        'tag': 'B',
        'statement': 'Return the highest common factor of two numbers without using any library gcd function.',
        'input': 'first = 128, second = 96',
    },
    {
        'tag': 'C',
        'statement': 'Given a list of temperatures, return the average of only the negative values.',
        'input': '[3, -2, -5, 0, 8, -1]',
    },
    {
        'tag': 'D',
        'statement': 'Check whether a sentence is a pangram containing every alphabet letter at least once.',
        'input': 'The quick brown fox jumps over a lazy dog',
    },
]

if not RUN_INFERENCE_EXAMPLES:
    print('RUN_INFERENCE_EXAMPLES=False, inference probe skipped.')
elif 'graph_decoder' not in globals():
    print('Stage E needs typed-IR decoder. Run Stage D first or set RUN_LOAD_SAVED_MODELS=True for compatible saved models.')
else:
    inf_texts = [build_problem_text(p['statement'], p['input']) for p in INFERENCE_PROBLEMS]
    inf_tags = [p['tag'] for p in INFERENCE_PROBLEMS]

    inf_emb = encode_texts(
        inf_texts,
        model_name=EMBED_MODEL,
        batch_size=EMBED_BATCH_SIZE,
        prompt=PROBLEM_PROMPT,
        cache_path=None,
    )
    inf_z = predict_embeddings(predictor, inf_emb)
    inf_norm = inf_z / np.maximum(np.linalg.norm(inf_z, axis=1, keepdims=True), 1e-12)
    cross = inf_norm @ inf_norm.T

    print('=' * 100)
    print('Predicted typed-IR latent cross-similarity matrix')
    display(pd.DataFrame(cross, index=inf_tags, columns=inf_tags).round(4))

    inf_retrieved_idx, inf_retrieved_scores = nearest_indices(
        inf_z,
        target_embeddings,
        top_k=RETRIEVAL_TOP_K,
        exclude_self=False,
    )

    def lexical_overlap(a, b):
        aw = set(str(a).lower().replace('\n', ' ').split())
        bw = set(str(b).lower().replace('\n', ' ').split())
        if not aw or not bw:
            return 0.0
        return len(aw & bw) / max(1, len(aw | bw))

    def choose_retrieved_ir_for_problem(row_id, ptext):
        best_idx = int(inf_retrieved_idx[row_id, 0])
        best_score = -1e9
        for rank, idx in enumerate(inf_retrieved_idx[row_id]):
            idx = int(idx)
            cosine = float(inf_retrieved_scores[row_id, rank])
            lexical = lexical_overlap(ptext, prepared.problem_texts[idx])
            function_bonus = 0.03 if ir_has_function(prepared.ir_texts[idx]) else 0.0
            score = cosine + 0.10 * lexical + function_bonus
            if score > best_score:
                best_score = score
                best_idx = idx
        return best_idx, f'combined cosine+lexical retrieval score={best_score:+.4f}'

    inf_decoder_sources = []
    chosen_ir_rows = []
    chosen_ir_reasons = []
    for row_id, ptext in enumerate(inf_texts):
        context_indices = rerank_context_indices(inf_retrieved_idx[row_id])
        inf_decoder_sources.append(build_decoder_source(ptext, context_indices))
        chosen_idx, reason = choose_retrieved_ir_for_problem(row_id, ptext)
        chosen_ir_rows.append(chosen_idx)
        chosen_ir_reasons.append(reason)

    decoded_irs = generate_graph_json(
        graph_decoder,
        inf_decoder_sources,
        inf_z,
        list(range(len(inf_texts))),
        max_source_length=GRAPH_MAX_SOURCE_LENGTH,
        max_new_tokens=GRAPH_MAX_NEW_TOKENS,
        source_prefix=DECODER_SOURCE_PREFIX,
    )

    generated_codes = {}
    for row_id, (problem, ptext, decoded_ir) in enumerate(zip(INFERENCE_PROBLEMS, inf_texts, decoded_irs)):
        tag = problem['tag']
        fallback_idx = int(chosen_ir_rows[row_id])
        final_ir = decoded_ir
        final_code = compile_ir_to_python(final_ir, function_name=f'{CODEGEN_FUNCTION_NAME}_{tag.lower()}')
        retrieved_baseline_code = compile_ir_to_python(
            prepared.ir_texts[fallback_idx],
            function_name=f'{CODEGEN_FUNCTION_NAME}_{tag.lower()}_retrieved_baseline',
        )
        generated_codes[tag] = final_code

        print('\n' + '=' * 100)
        print(f'PROBLEM {tag}: {ptext}')
        print('-' * 100)
        print('--- top retrieved typed IR prototypes from predicted latent ---')
        for rank in range(min(RETRIEVAL_PRINT_K, inf_retrieved_idx.shape[1])):
            idx = int(inf_retrieved_idx[row_id, rank])
            print(f"rank {rank + 1}: row={idx} cosine={inf_retrieved_scores[row_id, rank]:+.4f}")
            print(prepared.problem_texts[idx][:360])
            print(prepared.ir_texts[idx][:900])
            print()

        print(f'chosen retrieved baseline: row={fallback_idx} ({chosen_ir_reasons[row_id]})')
        print('\n--- decoder input prefix ---')
        print(inf_decoder_sources[row_id][:1200])
        print('\n--- decoded typed IR from problem + retrieved IR + predicted latent ---')
        print(str(decoded_ir)[:1800])
        print('valid typed IR:', is_valid_typed_ir(decoded_ir))
        print('\n--- retrieved baseline code (diagnostic only, not final output) ---')
        print(retrieved_baseline_code)
        print('\n--- final predicted code solution (typed-IR compiler, no final LLM) ---')
        print(final_code)

    def run_probe_tests(code_by_tag):
        tests = {
            'A': [((84, 30), 6), ((17, 13), 1)],
            'B': [((128, 96), 32), ((21, 14), 7)],
            'C': [(([3, -2, -5, 0, 8, -1],), (-2 - 5 - 1) / 3), (([1, 2, 3],), 0)],
            'D': [(('The quick brown fox jumps over a lazy dog',), True), (('hello world',), False)],
        }
        for tag, cases in tests.items():
            namespace = {}
            try:
                exec(code_by_tag[tag], namespace)
                fn = namespace[f'{CODEGEN_FUNCTION_NAME}_{tag.lower()}']
            except Exception as exc:
                print(f'Probe {tag}: could not load generated code:', repr(exc))
                continue
            print('\nProbe', tag)
            for args, expected in cases:
                try:
                    got = fn(*args)
                    ok = abs(got - expected) < 1e-6 if isinstance(expected, float) else got == expected
                    print(' args=', args, 'expected=', expected, 'got=', got, 'ok=', ok)
                except Exception as exc:
                    print(' args=', args, 'expected=', expected, 'raised=', repr(exc))

    print('\n' + '=' * 100)
    print('Execution probes for final decoded-IR code')
    run_probe_tests(generated_codes)
